# M5 Forecasting Accuracy — EDA
**Walmart の商品別日次販売数を予測する時系列コンペ**

- 30,490 商品 × 1,969 日 (2011-01-29 〜 2016-06-19)
- 3州 (CA, TX, WI) × 10店舗 × 3カテゴリ (HOBBIES, HOUSEHOLD, FOODS)
- 評価指標: **WRMSSE** (Weighted Root Mean Squared Scaled Error)
- 予測対象: 最後28日間 (d_1914〜d_1941)

### データ構成
| ファイル | 行数 | 内容 |
|----------|------|------|
| sales_train_validation.csv | 30,490 | d_1〜d_1913 の販売数 (wide形式) |
| sales_train_evaluation.csv | 30,490 | d_1〜d_1941 の販売数 (評価用、+28日) |
| calendar.csv | 1,969 | 日付、曜日、イベント、SNAP情報 |
| sell_prices.csv | 6,841,121 | 店舗×商品×週の販売価格 |
| sample_submission.csv | 60,980 | validation + evaluation の提出フォーマット |

In [ ]:
# ============================================================
# SETUP CELL — 環境・認証・データ確認・読み込み
# ============================================================
import sys, os, shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# ============================================================
# [CONFIG] 手動設定 (自動検出する場合は None のまま)
# ============================================================
USER_DATA_DIR = None   # 例: '/content/drive/MyDrive/m5'

# ============================================================
# [1] 環境検出
# ============================================================
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
print(f"[1] Environment : {'Google Colab' if IS_COLAB else 'Local'}")

COMPETITION = 'm5-forecasting-accuracy'
DATA_FILES  = ['calendar.csv', 'sales_train_evaluation.csv',
               'sell_prices.csv', 'sample_submission.csv']

def has_all_files(d: Path) -> bool:
    return d.exists() and all((d / f).exists() for f in DATA_FILES)

# ============================================================
# [2] DATA_DIR の決定
# ============================================================
DATA_DIR = None

if USER_DATA_DIR is not None:
    DATA_DIR = Path(USER_DATA_DIR)

elif IS_COLAB:
    # 2a. /content/ をチェック (通信なし)
    for c in [Path(f'/content/{COMPETITION}'), Path('/content/data')]:
        if has_all_files(c):
            DATA_DIR = c
            break

    # 2b. Drive をマウントして確認 (通信1回、失敗時はスキップ)
    if DATA_DIR is None:
        _drive_ok = Path('/content/drive/MyDrive').exists()
        if not _drive_ok:
            try:
                print("[2] Mounting Google Drive...")
                from google.colab import drive
                drive.mount('/content/drive', force_remount=False)
                _drive_ok = True
            except Exception as _e:
                print(f"[2] Drive mount failed: {_e}")
        if _drive_ok:
            for c in [
                Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}'),
                Path(f'/content/drive/MyDrive/{COMPETITION}'),
            ]:
                if has_all_files(c):
                    DATA_DIR = c
                    break

                # 2c. Drive にも見つからない → Kaggle API でダウンロード
    if DATA_DIR is None:
        DATA_DIR = Path(f'/content/{COMPETITION}')
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        # access_token を Drive から取得
        _token_path = Path('/content/drive/MyDrive/.kaggle/access_token')
        if not _token_path.exists():
            raise FileNotFoundError(
                "Google Drive の マイドライブ/.kaggle/access_token が見つかりません"
            )
        _token = _token_path.read_text().strip()
        print(f"[2] access_token : {_token[:8]}... (from Drive)")

        # kaggle CLI は使わず requests で直接ダウンロード
        import requests, zipfile
        _url = f'https://www.kaggle.com/api/v1/competitions/data/download-all/{COMPETITION}'
        print(f"[3] Downloading from Kaggle API...")
        with requests.get(_url, headers={'Authorization': f'Bearer {_token}'},
                          stream=True, allow_redirects=True) as _r:
            if _r.status_code == 401:
                raise RuntimeError("401 Unauthorized — access_token が無効です")
            if _r.status_code == 403:
                raise RuntimeError("403 Forbidden — コンペのルール同意が必要です")
            _r.raise_for_status()
            _zip = DATA_DIR / f'{COMPETITION}.zip'
            _total = int(_r.headers.get('content-length', 0))
            _done = 0
            with open(_zip, 'wb') as _f:
                for _chunk in _r.iter_content(1024 * 1024):
                    _f.write(_chunk)
                    _done += len(_chunk)
                    if _total:
                        print(f"\r    {_done/1e6:.0f} / {_total/1e6:.0f} MB", end='')
        print()
        with zipfile.ZipFile(_zip) as _z:
            _z.extractall(DATA_DIR)
        _zip.unlink()
        del _token

else:
    # ローカル: 通信なし、データは手動管理
    for c in [
        Path('/mnt/c/Users/hiroyuki_nakayama/src/kaggle-with-mcp/m5-forecasting-accuracy'),
        Path('.'),
    ]:
        if has_all_files(c):
            DATA_DIR = c
            break

# ============================================================
# [3] ファイル確認
# ============================================================
assert DATA_DIR is not None, "DATA_DIR が未設定です"
assert has_all_files(DATA_DIR), \
    f"ファイルが揃っていません: {[f for f in DATA_FILES if not (DATA_DIR/f).exists()]}"

print(f"[2] DATA_DIR    : {DATA_DIR}")
for f in DATA_FILES:
    size = (DATA_DIR / f).stat().st_size / 1e6
    print(f"    {f:<40} {size:6.1f} MB")

# ============================================================
# [4] データ読み込み
# ============================================================
FIG_DIR = DATA_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR = str(DATA_DIR) + '/'
FIG_DIR  = str(FIG_DIR) + '/'

print("[3] Loading data...")
calendar = pd.read_csv(DATA_DIR + 'calendar.csv', parse_dates=['date'])
sales    = pd.read_csv(DATA_DIR + 'sales_train_evaluation.csv')
prices   = pd.read_csv(DATA_DIR + 'sell_prices.csv')

print(f"    calendar : {calendar.shape}")
print(f"    sales    : {sales.shape}")
print(f"    prices   : {prices.shape}")
print("✓ Ready")

In [ ]:
# --- 基本統計 ---
print("=== calendar ===")
print(f"期間: {calendar['date'].min()} ~ {calendar['date'].max()}")
print(f"日数: {len(calendar)}")
print(f"\nイベント数: {calendar['event_name_1'].notna().sum()} 日 (event_1), {calendar['event_name_2'].notna().sum()} 日 (event_2)")
print(f"\nイベント種類:")
print(calendar['event_type_1'].value_counts())

print("\n=== sales ===")
meta_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
d_cols = [c for c in sales.columns if c.startswith('d_')]
print(f"商品数: {len(sales)}")
print(f"日数: {len(d_cols)} (d_1 ~ d_{len(d_cols)})")
print(f"\n州別商品数:")
print(sales['state_id'].value_counts())
print(f"\nカテゴリ別商品数:")
print(sales['cat_id'].value_counts())
print(f"\n店舗別商品数:")
print(sales['store_id'].value_counts())

print("\n=== sell_prices ===")
print(f"ユニーク商品数: {prices['item_id'].nunique()}")
print(f"ユニーク店舗数: {prices['store_id'].nunique()}")
print(f"価格範囲: ${prices['sell_price'].min():.2f} ~ ${prices['sell_price'].max():.2f}")
print(f"価格中央値: ${prices['sell_price'].median():.2f}")
print(f"週数: {prices['wm_yr_wk'].nunique()}")

## Step 1: 基本統計と時系列の全体像

In [ ]:
# --- Step 1a: 全期間の合計売上 時系列プロット ---
# d_cols の合計を日別に算出
daily_total = sales[d_cols].sum(axis=0).values  # shape: (1941,)
dates = calendar['date'].values[:len(daily_total)]

fig, ax = plt.subplots(figsize=(18, 5))
ax.plot(dates, daily_total, linewidth=0.5, alpha=0.8, color='steelblue')
ax.set_title('Total Daily Sales (All Items, All Stores)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Total Units Sold')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 年末年始を強調
for year in range(2011, 2017):
    xmas = pd.Timestamp(f'{year}-12-25')
    if xmas >= pd.Timestamp(dates[0]) and xmas <= pd.Timestamp(dates[-1]):
        ax.axvline(xmas, color='red', alpha=0.3, linestyle='--', linewidth=0.8)

ax.legend(['Daily Total', 'Christmas'], loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR + '01_total_daily_sales.png', dpi=150)
plt.show()

print(f"売上範囲: {daily_total.min():,.0f} ~ {daily_total.max():,.0f}")
print(f"売上平均: {daily_total.mean():,.0f} ± {daily_total.std():,.0f}")

In [ ]:
# --- Step 1b: 曜日別・月別の平均売上 ---
cal_sales = pd.DataFrame({
    'date': dates,
    'total': daily_total,
})
cal_sales = cal_sales.merge(calendar[['date', 'weekday', 'wday', 'month', 'year']], on='date')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 曜日別
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_stats = cal_sales.groupby('weekday')['total'].agg(['mean', 'std']).reindex(weekday_order)
axes[0].bar(range(7), weekday_stats['mean'], yerr=weekday_stats['std'], 
            color=['#4C72B0' if d not in ['Saturday', 'Sunday'] else '#DD8452' for d in weekday_order],
            capsize=3, alpha=0.8)
axes[0].set_xticks(range(7))
axes[0].set_xticklabels([d[:3] for d in weekday_order])
axes[0].set_title('Average Daily Sales by Weekday')
axes[0].set_ylabel('Total Units Sold')

# 月別
month_stats = cal_sales.groupby('month')['total'].agg(['mean', 'std'])
axes[1].bar(range(1, 13), month_stats['mean'], yerr=month_stats['std'],
            color='steelblue', capsize=3, alpha=0.8)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_title('Average Daily Sales by Month')
axes[1].set_ylabel('Total Units Sold')

plt.tight_layout()
plt.savefig(FIG_DIR + '02_weekday_month_sales.png', dpi=150)
plt.show()

# 曜日の売上比
print("曜日別 平均売上 (週平均=1.0 基準):")
weekday_mean = cal_sales.groupby('weekday')['total'].mean().reindex(weekday_order)
for d, v in zip(weekday_order, weekday_mean / weekday_mean.mean()):
    print(f"  {d:>10}: {v:.3f}")

In [ ]:
# --- Step 1c: イベント日の売上変動 ---
cal_with_sales = calendar[['date', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']].copy()
cal_with_sales = cal_with_sales[cal_with_sales['d'].isin([f'd_{i}' for i in range(1, len(daily_total)+1)])]
cal_with_sales['total'] = daily_total

# イベント有無別
event_days = cal_with_sales[cal_with_sales['event_name_1'].notna()]
non_event_days = cal_with_sales[cal_with_sales['event_name_1'].isna()]

print(f"イベント日: {len(event_days)} 日, 平均売上: {event_days['total'].mean():,.0f}")
print(f"非イベント日: {len(non_event_days)} 日, 平均売上: {non_event_days['total'].mean():,.0f}")
print(f"差異: {(event_days['total'].mean() / non_event_days['total'].mean() - 1)*100:+.1f}%")

# イベント種別ごとの売上
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# イベントタイプ別
event_type_sales = event_days.groupby('event_type_1')['total'].agg(['mean', 'count'])
event_type_sales = event_type_sales.sort_values('mean', ascending=True)
colors = {'Cultural': '#4C72B0', 'National': '#DD8452', 'Religious': '#55A868', 'Sporting': '#C44E52'}
bar_colors = [colors.get(t, 'gray') for t in event_type_sales.index]
axes[0].barh(event_type_sales.index, event_type_sales['mean'], color=bar_colors, alpha=0.8)
axes[0].axvline(non_event_days['total'].mean(), color='black', linestyle='--', label='Non-event avg')
for i, (idx, row) in enumerate(event_type_sales.iterrows()):
    axes[0].text(row['mean'] + 500, i, f'n={int(row["count"])}', va='center', fontsize=10)
axes[0].set_title('Average Sales by Event Type')
axes[0].set_xlabel('Total Units Sold')
axes[0].legend()

# 主要イベント別 (出現2回以上)
event_name_sales = event_days.groupby('event_name_1')['total'].agg(['mean', 'count'])
event_name_sales = event_name_sales[event_name_sales['count'] >= 2].sort_values('mean', ascending=True)
axes[1].barh(range(len(event_name_sales)), event_name_sales['mean'], color='steelblue', alpha=0.8)
axes[1].set_yticks(range(len(event_name_sales)))
axes[1].set_yticklabels(event_name_sales.index, fontsize=9)
axes[1].axvline(non_event_days['total'].mean(), color='black', linestyle='--', label='Non-event avg')
axes[1].set_title('Average Sales by Event Name (n≥2)')
axes[1].set_xlabel('Total Units Sold')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '03_event_sales.png', dpi=150)
plt.show()

# クリスマス・サンクスギビングの売上を具体的に
for event_name in ['Christmas', 'Thanksgiving', 'SuperBowl', 'Easter']:
    ev = event_days[event_days['event_name_1'] == event_name]
    if len(ev) > 0:
        print(f"\n{event_name}: 平均売上 {ev['total'].mean():,.0f} ({(ev['total'].mean()/non_event_days['total'].mean()-1)*100:+.1f}% vs 非イベント)")
        for _, row in ev.iterrows():
            print(f"  {row['date'].strftime('%Y-%m-%d')}: {row['total']:,.0f}")

## Step 2: 階層構造（Hierarchy）の分析
州別（CA, TX, WI）、カテゴリ別（FOODS, HOBBIES, HOUSEHOLD）の売上比率と店舗ごとの特性

In [ ]:
# --- Step 2a: 州別・カテゴリ別の売上比率 ---
total_per_item = sales[d_cols].sum(axis=1)
sales['total_sales'] = total_per_item

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 州別
state_sales = sales.groupby('state_id')['total_sales'].sum()
state_colors = {'CA': '#4C72B0', 'TX': '#DD8452', 'WI': '#55A868'}
axes[0].pie(state_sales, labels=state_sales.index, autopct='%1.1f%%', 
            colors=[state_colors[s] for s in state_sales.index], startangle=90)
axes[0].set_title('Sales by State')

# カテゴリ別
cat_sales = sales.groupby('cat_id')['total_sales'].sum()
cat_colors = {'FOODS': '#C44E52', 'HOBBIES': '#8172B2', 'HOUSEHOLD': '#CCB974'}
axes[1].pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%',
            colors=[cat_colors[c] for c in cat_sales.index], startangle=90)
axes[1].set_title('Sales by Category')

# 店舗別
store_sales = sales.groupby('store_id')['total_sales'].sum().sort_values(ascending=True)
colors_store = [state_colors[s.split('_')[0]] for s in store_sales.index]
axes[2].barh(store_sales.index, store_sales.values, color=colors_store, alpha=0.8)
axes[2].set_title('Total Sales by Store')
axes[2].set_xlabel('Total Units Sold')

plt.tight_layout()
plt.savefig(FIG_DIR + '04_hierarchy_sales.png', dpi=150)
plt.show()

# 数値でも表示
print("州×カテゴリ 売上構成比:")
cross = sales.groupby(['state_id', 'cat_id'])['total_sales'].sum().unstack()
print((cross / cross.sum().sum() * 100).round(1))

In [ ]:
# --- Step 2b: 州別の日次売上推移 ---
fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

for i, state in enumerate(['CA', 'TX', 'WI']):
    state_mask = sales['state_id'] == state
    for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
        cat_mask = sales['cat_id'] == cat
        daily = sales.loc[state_mask & cat_mask, d_cols].sum(axis=0).values
        # 7日移動平均でスムージング
        daily_smooth = pd.Series(daily).rolling(7, center=True).mean().values
        axes[i].plot(dates, daily_smooth, linewidth=0.8, alpha=0.8, 
                     color=cat_colors[cat], label=cat)
    axes[i].set_title(f'{state}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Daily Sales (7d MA)')
    axes[i].legend(loc='upper left')
    axes[i].xaxis.set_major_locator(mdates.YearLocator())
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Daily Sales by State × Category (7-day Moving Average)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + '05_state_category_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Step 2c: 特定1店舗の深掘り (CA_3 = 最大売上店舗) ---
# まず最大売上店舗を特定
top_store = sales.groupby('store_id')['total_sales'].sum().idxmax()
print(f"最大売上店舗: {top_store}")

store_data = sales[sales['store_id'] == top_store]

fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# dept_id 別の推移
dept_sales = store_data.groupby('dept_id')[d_cols].sum()
dept_total = dept_sales.sum(axis=1).sort_values(ascending=False)
top_depts = dept_total.head(6).index

for dept in top_depts:
    daily = dept_sales.loc[dept].values
    daily_smooth = pd.Series(daily).rolling(28, center=True).mean().values
    axes[0].plot(dates, daily_smooth, linewidth=1.0, alpha=0.8, label=dept)

axes[0].set_title(f'{top_store}: Daily Sales by Department (28d MA)', fontsize=13)
axes[0].set_ylabel('Daily Sales')
axes[0].legend(loc='upper left', ncol=3)

# cat_id 別の積み上げ (月次集計)
monthly = store_data.copy()
# 月次に変換
cat_monthly = {}
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    daily_vals = store_data[store_data['cat_id'] == cat][d_cols].sum(axis=0).values
    cat_monthly[cat] = daily_vals

# 28日ごとに集計して積み上げ
period = 28
n_periods = len(daily_total) // period
x_dates = dates[::period][:n_periods]
bottom = np.zeros(n_periods)

for cat, color in cat_colors.items():
    vals = np.array([cat_monthly[cat][i*period:(i+1)*period].sum() for i in range(n_periods)])
    axes[1].bar(x_dates, vals, bottom=bottom, width=20, color=color, alpha=0.8, label=cat)
    bottom += vals

axes[1].set_title(f'{top_store}: Sales by Category (4-week periods, stacked)', fontsize=13)
axes[1].set_ylabel('Total Units Sold')
axes[1].legend(loc='upper left')
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(FIG_DIR + '06_store_deep_dive.png', dpi=150)
plt.show()

# 部門別構成比
print(f"\n{top_store} 部門別売上構成比:")
for dept, total in dept_total.items():
    print(f"  {dept:>15}: {total:>12,.0f} ({total/dept_total.sum()*100:.1f}%)")

## Step 3: 価格（Price）と在庫の動向
sell_priceの時間経過による変化、価格変更と売上数量の相関

In [ ]:
# --- Step 3a: 価格分布とカテゴリ別価格帯 ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 全体の価格分布
axes[0].hist(prices['sell_price'], bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Sell Price Distribution (All Items)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, 30)  # 大部分は$30以下

# カテゴリ別の価格分布 (item_idからcat_idを取得)
prices_with_cat = prices.merge(sales[['item_id', 'cat_id']].drop_duplicates(), on='item_id')
for cat, color in cat_colors.items():
    cat_prices = prices_with_cat[prices_with_cat['cat_id'] == cat]['sell_price']
    axes[1].hist(cat_prices, bins=80, alpha=0.5, color=color, label=f'{cat} (med=${cat_prices.median():.2f})')

axes[1].set_title('Price Distribution by Category')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0, 30)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '07_price_distribution.png', dpi=150)
plt.show()

print("カテゴリ別 価格統計:")
print(prices_with_cat.groupby('cat_id')['sell_price'].describe().round(2))

In [ ]:
# --- Step 3b: サンプル商品の価格推移 ---
# 売上上位の商品から各カテゴリ2つずつサンプル
np.random.seed(42)
sample_items = []
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    cat_items = sales[sales['cat_id'] == cat].nlargest(20, 'total_sales')['item_id'].values
    sample_items.extend(np.random.choice(cat_items, 2, replace=False))

# wm_yr_wk → 日付マッピング (週の最初の日)
wk_to_date = calendar.groupby('wm_yr_wk')['date'].first().to_dict()

fig, axes = plt.subplots(3, 2, figsize=(16, 10))
axes = axes.flatten()

for i, item in enumerate(sample_items):
    item_prices = prices[(prices['item_id'] == item) & (prices['store_id'] == top_store)]
    if len(item_prices) == 0:
        item_prices = prices[prices['item_id'] == item].groupby('wm_yr_wk')['sell_price'].mean().reset_index()
    
    item_prices = item_prices.copy()
    item_prices['date'] = item_prices['wm_yr_wk'].map(wk_to_date)
    item_prices = item_prices.dropna(subset=['date']).sort_values('date')
    
    cat = item.split('_')[0]
    axes[i].plot(item_prices['date'], item_prices['sell_price'], 
                 color=cat_colors[cat], linewidth=1.0, alpha=0.8)
    axes[i].set_title(f'{item}', fontsize=10)
    axes[i].set_ylabel('Price ($)')
    
    # 価格変更ポイントをマーク
    price_changes = item_prices['sell_price'].diff().ne(0) & item_prices['sell_price'].diff().notna()
    n_changes = price_changes.sum()
    axes[i].text(0.02, 0.95, f'{n_changes} price changes', transform=axes[i].transAxes, 
                 fontsize=9, va='top')

plt.suptitle(f'Price History of Sample Items ({top_store})', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + '08_price_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Step 3c: 価格変更と売上の相関 ---
# サンプル: 売上上位100商品 × top_store で価格変更の影響を集計
top_items = sales[sales['store_id'] == top_store].nlargest(100, 'total_sales')

# 週次の売上を作る (wm_yr_wk単位)
wk_to_dcols = {}
for _, row in calendar[calendar['d'].isin(d_cols)].iterrows():
    wk = row['wm_yr_wk']
    if wk not in wk_to_dcols:
        wk_to_dcols[wk] = []
    wk_to_dcols[wk].append(row['d'])

# 各商品の価格変更前後の売上変化を集計
price_change_effects = []

for _, item_row in top_items.iterrows():
    item_id = item_row['item_id']
    item_prices = prices[(prices['item_id'] == item_id) & (prices['store_id'] == top_store)].sort_values('wm_yr_wk')
    
    if len(item_prices) < 3:
        continue
    
    item_prices = item_prices.copy()
    item_prices['price_pct_change'] = item_prices['sell_price'].pct_change()
    
    # 週別売上を計算
    weekly_sales = []
    for _, pr in item_prices.iterrows():
        wk = pr['wm_yr_wk']
        if wk in wk_to_dcols:
            cols_in_wk = [c for c in wk_to_dcols[wk] if c in d_cols]
            if cols_in_wk:
                wk_sale = item_row[cols_in_wk].sum()
                weekly_sales.append(wk_sale)
            else:
                weekly_sales.append(np.nan)
        else:
            weekly_sales.append(np.nan)
    
    item_prices['weekly_sales'] = weekly_sales
    item_prices['sales_pct_change'] = item_prices['weekly_sales'].pct_change()
    
    # 価格変更があった週のみ
    changed = item_prices[item_prices['price_pct_change'].abs() > 0.01].dropna(subset=['sales_pct_change'])
    for _, c in changed.iterrows():
        if np.isfinite(c['price_pct_change']) and np.isfinite(c['sales_pct_change']):
            price_change_effects.append({
                'price_change_pct': c['price_pct_change'] * 100,
                'sales_change_pct': c['sales_pct_change'] * 100,
                'cat': item_id.split('_')[0]
            })

effects_df = pd.DataFrame(price_change_effects)
# 外れ値除去 (±200%)
effects_df = effects_df[(effects_df['sales_change_pct'].abs() < 200) & (effects_df['price_change_pct'].abs() < 50)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 散布図
for cat, color in cat_colors.items():
    mask = effects_df['cat'] == cat
    axes[0].scatter(effects_df.loc[mask, 'price_change_pct'], 
                    effects_df.loc[mask, 'sales_change_pct'],
                    alpha=0.3, s=20, color=color, label=cat)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_xlabel('Price Change (%)')
axes[0].set_ylabel('Sales Change (%)')
axes[0].set_title('Price Change vs Sales Change (Weekly)')
axes[0].legend()

# 値下げ/値上げ時の売上変化の分布
price_down = effects_df[effects_df['price_change_pct'] < -1]['sales_change_pct']
price_up = effects_df[effects_df['price_change_pct'] > 1]['sales_change_pct']

axes[1].hist(price_down, bins=50, alpha=0.6, color='green', label=f'Price Down (n={len(price_down)}, med={price_down.median():+.1f}%)')
axes[1].hist(price_up, bins=50, alpha=0.6, color='red', label=f'Price Up (n={len(price_up)}, med={price_up.median():+.1f}%)')
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Sales Change (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Sales Response to Price Changes')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '09_price_sales_correlation.png', dpi=150)
plt.show()

corr = effects_df['price_change_pct'].corr(effects_df['sales_change_pct'])
print(f"価格変更率 vs 売上変更率 相関係数: {corr:.3f}")
print(f"値下げ時の売上変化中央値: {price_down.median():+.1f}%")
print(f"値上げ時の売上変化中央値: {price_up.median():+.1f}%")

## Step 4: 間欠需要（Zero Sales）の特定
売上0の日の割合、「発売前で0」と「発売後に0」の区別、0連続日数の分布

In [ ]:
# --- Step 4a: 売上0の全体像 ---
sales_matrix = sales[d_cols].values  # shape: (30490, 1941)

total_cells = sales_matrix.size
zero_cells = (sales_matrix == 0).sum()
zero_pct = zero_cells / total_cells * 100

print(f"全セル数: {total_cells:,}")
print(f"売上0のセル数: {zero_cells:,} ({zero_pct:.1f}%)")
print(f"売上>0のセル数: {total_cells - zero_cells:,} ({100-zero_pct:.1f}%)")

# 商品別の0比率
zero_pct_per_item = (sales_matrix == 0).sum(axis=1) / sales_matrix.shape[1] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(zero_pct_per_item, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribution of Zero-Sales Rate per Item')
axes[0].set_xlabel('% of Days with Zero Sales')
axes[0].set_ylabel('Number of Items')
axes[0].axvline(zero_pct_per_item.mean(), color='red', linestyle='--', label=f'Mean={zero_pct_per_item.mean():.1f}%')
axes[0].legend()

# カテゴリ別
for cat, color in cat_colors.items():
    mask = sales['cat_id'] == cat
    axes[1].hist(zero_pct_per_item[mask.values], bins=50, alpha=0.5, color=color, label=cat)
axes[1].set_title('Zero-Sales Rate by Category')
axes[1].set_xlabel('% of Days with Zero Sales')
axes[1].set_ylabel('Number of Items')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '10_zero_sales_distribution.png', dpi=150)
plt.show()

print(f"\n商品別 0比率統計:")
print(f"  平均: {zero_pct_per_item.mean():.1f}%")
print(f"  中央値: {np.median(zero_pct_per_item):.1f}%")
print(f"  0比率>90%の商品: {(zero_pct_per_item > 90).sum():,} ({(zero_pct_per_item > 90).sum()/len(zero_pct_per_item)*100:.1f}%)")
print(f"  0比率>50%の商品: {(zero_pct_per_item > 50).sum():,} ({(zero_pct_per_item > 50).sum()/len(zero_pct_per_item)*100:.1f}%)")

In [ ]:
# --- Step 4b: 「発売前で0」vs「発売後に0」の判別 ---
# 各商品の「初回販売日」= 最初に売上>0になった日のインデックス
first_sale_idx = np.argmax(sales_matrix > 0, axis=1)  # 最初の非ゼロのインデックス

# 初回販売がない(全部0)の商品
never_sold = (sales_matrix.sum(axis=1) == 0)
print(f"一度も売れていない商品: {never_sold.sum()} ({never_sold.sum()/len(never_sold)*100:.2f}%)")

# 発売前の0と発売後の0を分離
pre_launch_zeros = 0
post_launch_zeros = 0
post_launch_total_days = 0

for i in range(len(sales_matrix)):
    if never_sold[i]:
        pre_launch_zeros += sales_matrix.shape[1]
        continue
    
    first = first_sale_idx[i]
    pre_launch_zeros += first  # 初回販売前の0
    post_launch = sales_matrix[i, first:]
    post_launch_zeros += (post_launch == 0).sum()
    post_launch_total_days += len(post_launch)

print(f"\n発売前の0: {pre_launch_zeros:,} ({pre_launch_zeros/zero_cells*100:.1f}% of all zeros)")
print(f"発売後の0: {post_launch_zeros:,} ({post_launch_zeros/zero_cells*100:.1f}% of all zeros)")
print(f"発売後の0比率: {post_launch_zeros/post_launch_total_days*100:.1f}% (発売後の全日数に対して)")

# 初回販売日の分布
fig, ax = plt.subplots(figsize=(16, 5))
first_sale_dates = [dates[idx] for idx, ns in zip(first_sale_idx, never_sold) if not ns]
ax.hist(first_sale_dates, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
ax.set_title('Distribution of First Sale Date')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Items')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(FIG_DIR + '11_first_sale_date.png', dpi=150)
plt.show()

# 初回販売日がd_1の商品 vs 途中参入
day1_items = (first_sale_idx == 0) & ~never_sold
print(f"\nd_1から存在する商品: {day1_items.sum()} ({day1_items.sum()/len(day1_items)*100:.1f}%)")
print(f"途中参入の商品: {(~day1_items & ~never_sold).sum()} ({(~day1_items & ~never_sold).sum()/len(day1_items)*100:.1f}%)")

In [ ]:
# --- Step 4c: 発売後の0連続日数の分布 ---
from itertools import groupby

# サンプリング: ランダム3000商品 (全商品は重いので)
np.random.seed(42)
sample_idx = np.random.choice(np.where(~never_sold)[0], size=min(3000, (~never_sold).sum()), replace=False)

zero_streak_lengths = []

for i in sample_idx:
    first = first_sale_idx[i]
    post_launch = sales_matrix[i, first:]
    
    # 連続0のストリーク長を計算
    for is_zero, group in groupby(post_launch == 0):
        if is_zero:
            streak_len = sum(1 for _ in group)
            zero_streak_lengths.append(streak_len)

zero_streaks = np.array(zero_streak_lengths)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 連続0日数の分布 (ヒストグラム)
axes[0].hist(zero_streaks[zero_streaks <= 60], bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribution of Consecutive Zero-Sales Streaks (≤60 days)')
axes[0].set_xlabel('Consecutive Zero Days')
axes[0].set_ylabel('Frequency')
axes[0].axvline(7, color='red', linestyle='--', alpha=0.5, label='1 week')
axes[0].axvline(28, color='orange', linestyle='--', alpha=0.5, label='4 weeks')
axes[0].legend()

# 累積分布
sorted_streaks = np.sort(zero_streaks)
axes[1].plot(sorted_streaks, np.arange(1, len(sorted_streaks)+1)/len(sorted_streaks), 
             color='steelblue', linewidth=1.5)
axes[1].set_title('CDF of Consecutive Zero-Sales Streaks')
axes[1].set_xlabel('Consecutive Zero Days')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].axhline(0.9, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlim(0, 200)

# パーセンタイルをアノテーション
for p, label in [(50, '50th'), (90, '90th'), (99, '99th')]:
    val = np.percentile(zero_streaks, p)
    axes[1].annotate(f'{label}: {val:.0f}d', xy=(val, p/100), fontsize=10,
                     xytext=(val+10, p/100-0.05),
                     arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.savefig(FIG_DIR + '12_zero_streak_distribution.png', dpi=150)
plt.show()

print(f"0連続日数の統計 (発売後, n={len(zero_streaks):,} streaks):")
print(f"  平均: {zero_streaks.mean():.1f} 日")
print(f"  中央値: {np.median(zero_streaks):.1f} 日")
print(f"  90th percentile: {np.percentile(zero_streaks, 90):.0f} 日")
print(f"  99th percentile: {np.percentile(zero_streaks, 99):.0f} 日")
print(f"  最大: {zero_streaks.max()} 日")

# 長期ゼロ (28日以上) の割合
long_zero = (zero_streaks >= 28).sum()
print(f"\n28日以上連続0: {long_zero:,} streaks ({long_zero/len(zero_streaks)*100:.1f}%)")
print(f"  → 在庫切れ or 廃番の可能性が高い")

In [ ]:
# --- Step 4d: 日別の0比率推移 (発売後のみ) ---
# 各日の「発売済み商品数」と「そのうち売上0の商品数」
active_items_per_day = np.zeros(len(d_cols))
zero_active_per_day = np.zeros(len(d_cols))

for i in range(len(sales_matrix)):
    if never_sold[i]:
        continue
    first = first_sale_idx[i]
    for j in range(first, len(d_cols)):
        active_items_per_day[j] += 1
        if sales_matrix[i, j] == 0:
            zero_active_per_day[j] += 1

zero_rate_per_day = zero_active_per_day / np.maximum(active_items_per_day, 1) * 100

fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# 発売済み商品数の推移
axes[0].plot(dates, active_items_per_day, linewidth=0.8, color='steelblue')
axes[0].set_title('Number of Active Items (Post-Launch) Over Time')
axes[0].set_ylabel('Active Items')

# 発売後0比率の推移
axes[1].plot(dates, zero_rate_per_day, linewidth=0.5, color='tomato', alpha=0.5)
# 28日移動平均
zero_rate_smooth = pd.Series(zero_rate_per_day).rolling(28, center=True).mean().values
axes[1].plot(dates, zero_rate_smooth, linewidth=1.5, color='red', label='28d MA')
axes[1].set_title('Zero-Sales Rate Among Active Items')
axes[1].set_ylabel('Zero-Sales Rate (%)')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(FIG_DIR + '13_zero_rate_timeseries.png', dpi=150)
plt.show()

print(f"発売後0比率: 平均 {zero_rate_per_day.mean():.1f}%, 最小 {zero_rate_per_day.min():.1f}%, 最大 {zero_rate_per_day.max():.1f}%")

## EDA サマリ

### Step 1: 時系列の全体像
- 全体の売上トレンド、曜日効果、月次季節性
- イベント日 (Christmas, Thanksgiving等) の売上変動パターン

### Step 2: 階層構造
- 州別 (CA/TX/WI)、カテゴリ別 (FOODS/HOBBIES/HOUSEHOLD) の構成比
- 店舗ごとの部門別売上推移

### Step 3: 価格と売上の関係
- カテゴリ別の価格帯
- 価格変更時の売上への影響 (価格弾力性)

### Step 4: 間欠需要
- 売上0が全データの大半を占める
- 「発売前の0」と「発売後の0」の判別
- 0連続日数の分布と長期ゼロの特定